In [6]:
import numpy as np
import pandas as pd
import faiss
import chromadb

In [5]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 97.5 MB/s eta 0:00:00


In [7]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [8]:
movie_index_path = "/content/drive/MyDrive/movie_index.faiss"
dataset_path = "/content/drive/MyDrive/rag_dataset.csv"

In [9]:
df = pd.read_csv(dataset_path)

print("Movies:", len(df))

Movies: 1000590


In [10]:
index = faiss.read_index(movie_index_path)

print("Embeddings:", index.ntotal)
print("Dimension:", index.d)

Embeddings: 1000590
Dimension: 384


In [11]:
client = chromadb.PersistentClient(
    path="/content/drive/MyDrive/movie_chroma_db"
)

collection = client.get_or_create_collection(
    name="movies"
)

print("Vector Database Ready")

Vector Database Ready


In [12]:
df.duplicated().sum()

np.int64(66978)

In [13]:
batch_size = 5000

for start in range(0, index.ntotal, batch_size):

    end = min(start + batch_size, index.ntotal)

    embeddings = index.reconstruct_batch(
        np.arange(start, end)
    )

    ids = [str(i) for i in range(start, end)]

    collection.add(
        ids=ids,
        embeddings=embeddings.tolist()
    )

    percent = (end / index.ntotal) * 100

    print(
        "Stored:",
        end,
        "/",
        index.ntotal,
        f"({percent:.2f}%)"
    )

Stored: 5000 / 1000590 (0.50%)
Stored: 10000 / 1000590 (1.00%)
Stored: 15000 / 1000590 (1.50%)


KeyboardInterrupt: 

In [14]:
print("Total Vectors in ChromaDB:", collection.count())

Total Vectors in ChromaDB: 20000


In [15]:
movie_index_path = "/content/drive/MyDrive/movie_index.faiss"

index = faiss.read_index(movie_index_path)

print("Total Embeddings:", index.ntotal)

Total Embeddings: 1000590


In [16]:
client = chromadb.PersistentClient(
    path="/content/drive/MyDrive/movie_chroma_db"
)

collection = client.get_or_create_collection(
    name="movies"
)

print("Already Stored:", collection.count())

Already Stored: 20000
